# v3.6 New Model: OWLv2 Open-Vocabulary Detection (Apache-2.0)

OWLv2 is the improved version of OWL-ViT with better accuracy and efficiency.
Uses `google/owlv2-base-patch16` via HuggingFace Transformers (Apache-2.0).

In [ ]:
from PIL import Image
import numpy as np

img = Image.new('RGB', (640, 480), color=(50, 100, 150))
print('Image:', img.size)

In [ ]:
try:
    from transformers import Owlv2Processor, Owlv2ForObjectDetection
    import torch, time

    model_id = 'google/owlv2-base-patch16'
    processor = Owlv2Processor.from_pretrained(model_id)
    model = Owlv2ForObjectDetection.from_pretrained(model_id)

    texts = [['a cat', 'a dog', 'a table']]
    inputs = processor(text=texts, images=img, return_tensors='pt')

    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model(**inputs)
    latency_ms = (time.perf_counter() - t0) * 1000

    target_sizes = torch.tensor([img.size[::-1]])
    results = processor.post_process_object_detection(outputs, threshold=0.1, target_sizes=target_sizes)[0]

    print(f'Detected {len(results["boxes"])} objects in {latency_ms:.1f} ms')
    for box, score, label in zip(results['boxes'], results['scores'], results['labels']):
        print(f'  [{texts[0][label]}] score={score:.3f}')
except Exception as e:
    print(f'Skipped (missing deps): {e}')